<a href="https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. Unit of analysis

One row represents the daily performance of one content item (`content_hash_id`)
for one client (`client_hash_id`) on one report date (`report_date`).

## Time window

This notebook analyzes the March 2026 partition (`month = '2026-03'`). This is a
mid-panel month, which follows the FlyRank recommendation to avoid using the
final month (June 2026) when developing data logic.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
""")

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
These are historical measurements that are available before making a prediction.

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

## Label / Proxy
This notebook does not define a final modeling label. A future performance metric (such as future clicks or engagement) would be used as the prediction target in later modeling tasks.

## Context
These fields identify each observation and are used for grouping or joining, not as model features.

- report_date
- client_hash_id
- content_hash_id
- month

## Excluded
- client_hash_id – identifier only; not used as a feature.
- content_hash_id – identifier only; used only for joins/grouping.
- gsc_data_available – used only to check data availability.
- ga4_data_available – used only to filter valid GA4 rows because GA4 metrics are zero-filled before data collection begins.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
    SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_unavailable_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┬──────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │ ga4_unavailable_rows │
│   int64    │       int128       │       int128       │        int128        │
├────────────┼────────────────────┼────────────────────┼──────────────────────┤
│    9841378 │            3611061 │             413966 │              6408671 │
└────────────┴────────────────────┴────────────────────┴──────────────────────┘

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification of the data contract

The following queries verify the data contract for the March 2026 partition.

- **Grain:** The grain check returned **0 duplicate rows**, confirming that each row represents one content item (`content_hash_id`) for one client (`client_hash_id`) on one report date (`report_date`).

- **Counts and time window:** The March 2026 partition contains **9,841,378 rows**, covering the period from **2026-03-01** to **2026-03-31**.

- **Availability:** Filtering with `ga4_data_available IS TRUE` leaves **413,966 usable rows**. This confirms that many rows do not yet have GA4 data available, so the availability flag should be used when working with GA4 metrics.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Verify the grain (should return 0 rows)

print("=== Grain Check ===")
print(con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""))

# Query 2: Verify counts and date window

print("\n=== Counts & Window ===")
print(con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
"""))

# Query 3: Verify availability (using IS TRUE)

print("\n=== Availability Check ===")
print(con.sql(f"""
SELECT
    COUNT(*) AS usable_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
  AND ga4_data_available IS TRUE
"""))

=== Grain Check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘


=== Counts & Window ===
┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘


=== Availability Check ===
┌─────────────┐
│ usable_rows │
│    int64    │
├─────────────┤
│      413966 │
└─────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    ga4_data_available,
    COUNT(*) AS rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
GROUP BY ga4_data_available
ORDER BY ga4_data_available
""")

┌────────────────────┬─────────┐
│ ga4_data_available │  rows   │
│      boolean       │  int64  │
├────────────────────┼─────────┤
│ false              │ 6408671 │
│ true               │  413966 │
│ NULL               │ 3018741 │
└────────────────────┴─────────┘

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.